# Cold Calling 2 - Análise de Dados

**Exercício (Adaptado de Anderson et al., 2021, p. 110):**
Preço das passagens aéreas para Atlanta e Salt Lake City

Aluno: Luís Felipe Fernandes
Disciplina: Análise de Dados para Decisões em Negócios
Professor: Humberto Elias Garcia Lopes


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, HTML

# ==============================================================================
# 1. DADOS (ANDERSON ET AL., 2021, P. 110, ADAPTADO)
# ==============================================================================
origens = ["Cincinnati", "Nova York", "Chicago", "Denver", "Los Angeles", "Seattle", "Detroit",
           "Filadélfia", "Washington, D.C.", "Miami", "São Francisco", "Las Vegas", "Phoenix", "Dallas"]

atlanta = pd.Series(
    [340.10, 321.60, 291.60, 339.60, 359.60, 384.60, 309.60, 415.60, 293.60, 249.60, 539.60, 455.60, 359.60, 333.90],
    index=origens, name="Atlanta")
salt_lake = pd.Series(
    [570.10, 354.60, 465.60, 219.60, 311.60, 297.60, 471.60, 618.40, 513.60, 523.20, 381.60, 159.60, 267.60, 458.60],
    index=origens, name="Salt Lake City")
destinos = {"Atlanta": atlanta, "Salt Lake City": salt_lake}


def moeda(v):
    return "US$ " + f"{v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def pct(v):
    return f"{v:.2f}".replace(".", ",") + "%"


# ==============================================================================
# 2. MEDIDAS: TENDÊNCIA CENTRAL, DISPERSÃO, QUARTIS E CERCAS (REGRA DE 1,5 x IQR)
# ==============================================================================
def medidas(s):
    q1, q2, q3 = s.quantile([0.25, 0.50, 0.75])
    iqr = q3 - q1
    inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    modas = s.mode().tolist()
    return {
        "media": s.mean(), "mediana": s.median(), "modas": modas, "amodal": len(modas) == len(s),
        "repeticoes": int(s.value_counts().max()), "desvio": s.std(), "cv": s.std() / s.mean() * 100,
        "q1": q1, "q2": q2, "q3": q3, "iqr": iqr, "inferior": inferior, "superior": superior,
        "outliers": s[(s < inferior) | (s > superior)],
        "ate_q2": int((s <= q2).sum()), "ate_q3": int((s <= q3).sum()),
    }


m = {nome: medidas(s) for nome, s in destinos.items()}


# ==============================================================================
# 3. FUNÇÃO: TABELA ESTILIZADA (ABNT + STORYTELLING)
# ==============================================================================
def tabela_estilizada(df, titulo, subtitulo, formato=None, destaque=None):
    legenda_html = (
        f"<div style='text-align: left; margin-bottom: 8px; font-family: Arial, sans-serif;'>"
        f"<strong style='font-size: 11pt; color: black;'>{titulo}</strong><br>"
        f"<span style='font-size: 9.5pt; font-style: italic; color: #555555;'>{subtitulo}</span>"
        f"</div>"
    )
    estilo = df.style
    if formato:
        estilo = estilo.format(formato)
    if destaque:
        coluna, funcao = destaque
        estilo = estilo.map(funcao, subset=[coluna])
    return (
        estilo
        .set_table_styles([
            {"selector": "", "props": [("background-color", "white")]},
            {"selector": "caption", "props": [("caption-side", "top"), ("text-align", "left"), ("padding", "10px 12px 6px 12px"), ("background-color", "white"), ("color", "black")]},
            {"selector": "thead", "props": [("border-top", "2px solid black"), ("border-bottom", "1px solid black"), ("background-color", "white")]},
            {"selector": "th", "props": [("text-align", "center"), ("font-weight", "bold"), ("background-color", "white"), ("color", "black"), ("padding", "6px 10px")]},
            {"selector": "tbody tr:last-child", "props": [("border-bottom", "2px solid black")]},
            {"selector": "td", "props": [("border", "none"), ("text-align", "right"), ("padding", "6px 12px"), ("background-color", "white"), ("color", "black")]},
            {"selector": "th.index_name", "props": [("text-align", "left"), ("font-weight", "bold"), ("background-color", "white"), ("color", "black")]},
            {"selector": "th.row_heading", "props": [("text-align", "left"), ("font-weight", "normal"), ("border", "none"), ("background-color", "white"), ("color", "black")]},
        ])
        .set_caption(legenda_html)
    )


def nota(texto):
    return HTML(f"<div style='font-family: Arial, sans-serif; font-size: 8.5pt; color: #555555; background-color: white; display: inline-block; padding: 4px 10px; margin: 4px 0 18px 0;'>{texto}</div>")


# ==============================================================================
# 4. FUNÇÃO: BOXPLOT COMPARATIVO COM STORYTELLING (KNAFLIC)
# ==============================================================================
def boxplot_comparativo(titulo, subtitulo, nome_arquivo):
    fig, ax = plt.subplots(figsize=(9, 6.4), dpi=200)
    fig.patch.set_facecolor("#FFFFFF")
    ax.set_facecolor("#FFFFFF")

    ax.boxplot(
        [s.values for s in destinos.values()], positions=[1, 2], widths=0.42, whis=1.5, patch_artist=True, showfliers=False,
        boxprops=dict(facecolor="#E2E8F0", edgecolor="#718096", linewidth=1.3),
        medianprops=dict(color="#1A365D", linewidth=2.6),
        whiskerprops=dict(color="#718096", linewidth=1.3), capprops=dict(color="#718096", linewidth=1.3))

    rng = np.random.default_rng(7)
    for pos, (nome, s) in zip([1, 2], destinos.items()):
        atipico = (s < m[nome]["inferior"]) | (s > m[nome]["superior"])
        cores = np.where(atipico, "#d95f02", "#A0AEC0")
        ax.scatter(pos + rng.uniform(-0.09, 0.09, len(s)), s.values, s=34, c=cores, zorder=3, edgecolors="none")
        ax.text(pos, 118, f"IQR = {moeda(m[nome]['iqr'])}\nCV = {pct(m[nome]['cv'])}", ha="center", fontsize=9.5, color="#2D3748")

    for cidade, valor in m["Atlanta"]["outliers"].items():
        ax.annotate(f"{cidade}\n{moeda(valor)} (outlier)", xy=(1.03, valor), xytext=(1.32, valor + 25),
                    arrowprops=dict(arrowstyle="->", color="#333333", lw=1), fontsize=9.5, color="#d95f02", fontweight="bold")
    sup = m["Atlanta"]["superior"]
    ax.plot([0.72, 1.28], [sup, sup], linestyle=":", color="#d95f02", linewidth=1.4)
    ax.text(0.72, sup + 9, f"cerca superior {moeda(sup)}", fontsize=8.5, color="#4A5568", va="bottom")

    ax.set_xticks([1, 2])
    ax.set_xticklabels(list(destinos), fontsize=11, color="#2D3748")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"US$ {int(v)}"))
    ax.set_ylim(100, 660)
    ax.set_xlim(0.5, 2.5)
    ax.grid(axis="y", linestyle=":", color="#E0E0E0", zorder=0)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color("#888888")
    ax.tick_params(axis="y", labelsize=9, colors="#4A5568")

    fig.text(0.09, 0.96, titulo, fontsize=12, fontweight="bold", color="#1A202C")
    fig.text(0.09, 0.925, subtitulo, fontsize=9, color="#718096")
    plt.subplots_adjust(top=0.87)
    plt.savefig(nome_arquivo, dpi=200, bbox_inches="tight")
    plt.show()


# ==============================================================================
# 5. TABELA 1: OS DADOS, COM O PREÇO ATÍPICO EM DESTAQUE
# ==============================================================================
tab1 = pd.DataFrame({"Atlanta": atlanta, "Salt Lake City": salt_lake})
tab1.index.name = "Cidade de origem"
sup_atl = m["Atlanta"]["superior"]
display(tabela_estilizada(
    tab1,
    "Tabela 1: Um Preço Fora da Curva: São Francisco Paga US$ 539,60 para Atlanta",
    "Preço da passagem por cidade de origem, em US$ (n = 14 cidades para cada destino)",
    formato=moeda,
    destaque=("Atlanta", lambda v: "background-color: #FCE8D5 !important; color: #B34700 !important; font-weight: bold;" if v > sup_atl else "")))
display(nota("Nota. Em destaque, o preço acima da cerca superior de Atlanta. Fonte: Anderson et al. (2021, p. 110), adaptado."))

# ==============================================================================
# 6. TABELA 2: TENDÊNCIA CENTRAL E DISPERSÃO
# ==============================================================================
linhas = ["Média", "Mediana", "Moda", "Desvio padrão (s)", "Coeficiente de variação (CV)"]
tab2 = pd.DataFrame(index=linhas, columns=list(destinos))
for nome in destinos:
    x = m[nome]
    moda_txt = "Amodal (nenhum preço se repete)" if x["amodal"] else f"{moeda(x['modas'][0])} ({x['repeticoes']} vezes)"
    tab2[nome] = [moeda(x["media"]), moeda(x["mediana"]), moda_txt, moeda(x["desvio"]), pct(x["cv"])]
tab2.index.name = "Medida"
display(tabela_estilizada(
    tab2,
    f"Tabela 2: Salt Lake City Custa Mais, mas o Que a Distingue É a Variabilidade: CV de {pct(m['Salt Lake City']['cv'])}",
    "Medidas de tendência central e de dispersão, em US$ (n = 14 cidades para cada destino)"))
display(nota("Nota. Desvio padrão amostral (divisor n - 1)."))

# ==============================================================================
# 7. TABELA 3: QUARTIS, CERCAS E OUTLIERS
# ==============================================================================
linhas = ["1º quartil (Q1)", "2º quartil (Q2, mediana)", "3º quartil (Q3)", "Amplitude interquartil (IQR)",
          "Cerca inferior (Q1 - 1,5 × IQR)", "Cerca superior (Q3 + 1,5 × IQR)", "Outliers",
          "Cidades com preço até Q2 (50%)", "Cidades com preço até Q3 (75%)"]
tab3 = pd.DataFrame(index=linhas, columns=list(destinos))
for nome in destinos:
    x = m[nome]
    outliers_txt = "; ".join(f"{c} ({moeda(v)})" for c, v in x["outliers"].items()) or "Nenhum"
    tab3[nome] = [moeda(x["q1"]), moeda(x["q2"]), moeda(x["q3"]), moeda(x["iqr"]), moeda(x["inferior"]), moeda(x["superior"]),
                  outliers_txt, f"{x['ate_q2']} de {len(destinos[nome])}", f"{x['ate_q3']} de {len(destinos[nome])}"]
tab3.index.name = "Medida"
display(tabela_estilizada(
    tab3,
    "Tabela 3: Só Atlanta Cruza a Cerca: O Outlier de São Francisco e a Ausência de Atípicos em Salt Lake City",
    "Quartis, amplitude interquartil e cercas pela regra de 1,5 × IQR, em US$ (n = 14 cidades para cada destino)"))
display(nota("Nota. Quartis calculados pelo método quantile do pandas (interpolação linear)."))

# ==============================================================================
# 8. GRÁFICO: BOXPLOT COMPARATIVO
# ==============================================================================
boxplot_comparativo(
    "Salt Lake City Tem Tarifas Mais Dispersas, Mas o Único Outlier Está em Atlanta",
    "Preço da passagem de 14 cidades de origem para cada destino, em US$. Cercas pela regra de 1,5 × IQR.",
    "boxplot_tarifas.png")
